# H=8 — when the probe row space is the WHOLE hidden state, does the edit land?

**The idea (Sevan, 2026-08-13).** Every negative in this thread has the same escape hatch: readout
injection writes inside a **4-dimensional** probe row space, and the direction a successful edit needs
provably lies mostly outside it (`delta_h_analysis`: row-space fraction 0.096 against a chance level of
0.125 — *at or below chance*). So the obvious question is what happens when that excuse is removed.

**At `H = 8` it can be removed exactly.** Fit a linear position probe (rank 4), project its row space out
of the state, refit on what remains (INLP, Ravfogel et al. 2020) — a second rank-4 probe, orthogonal to
the first by construction. **4 + 4 = 8 = H.** The two probes together span the entire hidden state, so:

* every direction is reachable — the reachable fraction of *any* target `Δh` is **exactly 1.0**;
* the combined map is **square**, so specifying both readouts determines the new state **uniquely** —
  there is no null space left, and "injection" stops being a minimal edit and becomes a complete state
  specification.

If "the edit direction is outside the row space" was the binding constraint, this is where the edit lands.

> **Why `H = 8` specifically.** It is the world's true state dimensionality (2 objects × position +
> velocity), and it is the smallest arm of the hidden-size sweep. The cost is that it is a **much weaker
> model** — next-step RMSE 0.1495 vs 0.1041 at H=256, position R² ≈ 0.17 vs 0.83 — so every number here
> must be read against **its own** unsteered row and its own oracle, never against the H=256 thread values.
> `CONTROL_RUNS.md` also records that structural editors at H=8/H=32 degrade badly, which is why the
> **fidelity ratio** does more work in this notebook than in any other.

**Companion work:** `../history_editing/` (same branch) — where the un-edited complement turns out to be
*observation-shaped* history. This notebook attacks the same negative from the opposite side: not "what is
in the complement" but "what if there is no complement at all".

In [ ]:
# [1] setup
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

ROOT = Path("/home/sevan/research/physically-implicit-modeling")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "notebooks/experiments/editability/history_editing"))

from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio
from history_tools import ray_centroid, waterfall_grid   # the ONE waterfall spec, shared
from pim.figures.theme import style_ax
from pim.simulator.renderer import render_frame
from pim.simulator.sim import SimConfig
from pim.world_models import load_checkpoint, load_dataset

DEVICE = "cuda"
N_OBJ = 2          # objects in the world
K = 15             # rollout steps scored after the edit
N_EDIT = 256       # held-out edit samples
N_BANK = 2000      # edits sequences used to fit the probes
T_MIN = 10         # probe rows: frames 10..ef-1 (pre-edit, belief converged)
N_HIST = 8         # history frames used by the observation-channel oracle

OI = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00",
      "purple": "#CC79A7", "sky": "#56B4E9", "yellow": "#F0E442", "grey": "#5a5a5a"}
torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__, "| device", DEVICE, "|", torch.cuda.get_device_name(0))

## Definitions

Metric names/formulas copied from `../METRICS_AND_EDITORS.md`, computed by
`scripts/editability_metrics.py`. Nothing is re-derived here.

| term | meaning |
|---|---|
| **GRU · 8 hidden · dataset 4** (`runs/controls/H8`) | The `H=8` arm of the hidden-size sweep: 8-unit GRU, 5,384 params, pure next-step MSE, same recipe and data as every other controls run. Registry row: `../controls/CONTROL_RUNS.md`. **A much weaker model than the thread's H=256 baseline** — read it against its own unsteered row. |
| **dataset 4** (`datasets/4_fixed_refl_inview`) | 2 objects, 40 frames, `obs_res = 128`, `obs_noise_std = 0.2`, `position_noise_std = 0.04`, velocity exactly constant, teleport at `ef = 20`. |
| **INLP** | Iterative Null-space Projection: fit a probe, remove its row space from the state, refit. Used here to *construct* a second probe orthogonal to the first — not to erase an attribute. |
| **probe 1 / probe 2** | the two rank-4 linear position probes. Probe 2 is fit on states with `row(W₁)` projected out, so `row(W₁) ⊥ row(W₂)` and together they span `R⁸`. |
| **`A`** | the stacked map `[W₁; W₂] : R⁸ → R⁸`, **square**. Its condition number decides whether "solve for the state that reads the target" is well-posed. |
| **δ (the demanded change)** | the edit target is `positions[ef]`, i.e. the probe is driven to read the post-teleport position of both objects. |

| metric | formula | units | better | notes |
|---|---|---|---|---|
| **position R²** | `1 − ‖Y − probe(h)‖²/‖Y − Ȳ‖²`, held-out **by sequence**, vs the train mean | — | ↑ | reported per probe; probe 2's is measured on the deflated states it was fit on. |
| **reachable fraction** | `‖P_row Δh_true‖ / ‖Δh_true‖` for the working oracle's edit direction | — | ↑ | the **hard ceiling** on any injection-style editor. Chance is `√(rank/H)`; with both probes rank = H so the ceiling is **1.0 by construction**. |
| **cos(write, Δh_true)** | cosine between the editor's `Δh` and the oracle's, per sample then averaged | — | ↑ | **the quantity this notebook is really about.** Reachability says *whether* the answer is available; this says whether the probe **points at it**. Report the angle too — cos 0.9 is 26°, not "90% right". |
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)` over **differing** rays, per sample then averaged | −1…+1 | ↑ | +1 = edited world, −1 = unedited, **≈0 = equidistant OR garbage** — at H=8 the garbage reading is the common one, so never read it without the fidelity ratio. |
| **Target / Ghost RMSE** | `RMSE(edited₀, gt_edited)` over that zone at step 0 | obs intensity | ↓ | |
| **GT-traj RMSE** | `mean_s RMSE(edited_s, clean_obs[ef+s])` | obs intensity | ↓ | |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(unsteered)` | ratio | ↓ | **> 1 = the edit left the rollout further from the truth than doing nothing.** Load-bearing here. |
| **‖Δh‖/‖h‖** | mean write size relative to the state norm | ratio | — | the confound this notebook must control: an ill-conditioned inverse buys index movement with magnitude. |

**Editors.** *Unsteered* · *Injection · probe 1 only* (min-norm, rank 4 of 8) · **Injection · BOTH probes →
target** (the literal proposal: both readouts driven to the post-teleport position; the square map makes the
state unique) · *truncated-SVD variants* keeping the top-k singular directions of `A` · *norm-matched*
versions of each, scaled to `‖Δh_true‖` so direction is compared at fixed magnitude · *random direction,
norm-matched* · **Observation history overwrite (oracle)**, `n = 8` rendered frames of the rigidly
translated world — the working editor and the reference direction.

In [ ]:
# [2] load, provenance, helpers
model, info = load_checkpoint(ROOT / "runs/controls/H8/best_model.pt", device=DEVICE)
model.eval()
H = model.hidden_size

bundle = load_dataset(ROOT / "datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
sim = test.config["dataset"]["sim"]
dt, EF, R = float(sim["dt"]), int(edits.edit_frame), int(sim["obs_res"])

display(Markdown(f"""
| provenance | value |
|---|---|
| model | GRU · **{H} hidden** · dataset 4 (`runs/controls/H8/best_model.pt`) |
| dataset | `datasets/4_fixed_refl_inview`, `obs_res` = {R} |
| sensing / world noise | {sim['obs_noise_std']} / {sim['position_noise_std']} |
| edit frame `ef` | {EF} |
| probe bank / edit samples | {N_BANK} / {N_EDIT} sequences |
"""))


@torch.no_grad()
def hidden_states(obs, batch=512):
    return np.concatenate([
        model.get_hidden_states(torch.from_numpy(obs[i:i + batch]).float().to(DEVICE)).cpu().numpy()
        for i in range(0, len(obs), batch)])


@torch.no_grad()
def warm_up_to(obs, frame):
    """Teacher-force obs[0..frame-1]; a rollout's step 0 then decodes frame `frame`."""
    o = torch.from_numpy(obs).float().to(DEVICE)
    state = None
    for t in range(frame):
        _, state = model.step(o[:, t], state)
    return model.flat_state(state)


@torch.no_grad()
def teacher_force(h_flat, frames):
    state = model.state_from_flat(h_flat)
    f = torch.from_numpy(np.ascontiguousarray(frames)).float().to(DEVICE)
    for t in range(f.shape[1]):
        _, state = model.step(f[:, t], state)
    return model.flat_state(state)


@torch.no_grad()
def rollout(h_flat, steps=K):
    state = model.state_from_flat(torch.as_tensor(np.asarray(h_flat)).float().to(DEVICE))
    out = [model.decode(state)]
    for _ in range(steps - 1):
        p, state = model.predict_step(state)
        out.append(p)
    return torch.stack(out, 1).cpu().numpy()


def lstsq_fit(X, Y):
    return np.linalg.lstsq(np.c_[X, np.ones(len(X))], Y, rcond=None)[0]

## §1 — The construction: two orthogonal probes that span the whole state

Fit probe 1 on the states; project `row(W₁)` out; fit probe 2 on the remainder. Three things are asserted
rather than assumed: each probe is **rank 4**, the two row spaces are **orthogonal**, and together they
span **all 8 dimensions**.

The diagnostic that decides whether the resulting editor is well-posed is the **condition number of the
stacked map `A = [W₁; W₂]`**. `A` being square means the target readouts determine the state uniquely —
but if `A` has a near-null singular direction, that unique state is reached only by an enormous write.

In [ ]:
# [3] INLP: probe 1, deflate, probe 2 — with the structural claims asserted
hb = hidden_states(edits.obs[:N_BANK])
T = hb.shape[1]
pos_b = edits.positions[:N_BANK, :T, :N_OBJ, :].reshape(N_BANK, T, 4).astype(np.float64)
rows_t = np.arange(T_MIN, EF)                      # pre-edit frames only
X = hb[:, rows_t].reshape(-1, H).astype(np.float64)
Y = pos_b[:, rows_t].reshape(-1, 4)
n_tr = int(0.8 * N_BANK) * len(rows_t)             # split BY SEQUENCE
mu = Y[:n_tr].mean(0)

Ws, bs, Qs, r2s = [], [], [], []
X_def = X.copy()
for step in range(2):
    A_ = lstsq_fit(X_def[:n_tr], Y[:n_tr])
    W, b_ = A_[:-1].T, A_[-1]
    pred = np.c_[X_def[n_tr:], np.ones(len(X_def) - n_tr)] @ A_
    r2s.append(float(1 - ((pred - Y[n_tr:]) ** 2).sum() / ((Y[n_tr:] - mu) ** 2).sum()))
    rank = np.linalg.matrix_rank(W)
    assert rank == 4, f"probe {step+1} has rank {rank}, expected 4"
    Q = np.linalg.svd(W, full_matrices=False)[2][:rank]      # orthonormal basis of row(W)
    Ws.append(W); bs.append(b_); Qs.append(Q)
    X_def = X_def - X_def @ Q.T @ Q                          # deflate

Q_both = np.vstack(Qs)
W_cat, b_cat = np.vstack(Ws), np.concatenate(bs)
orth = float(np.abs(Qs[0] @ Qs[1].T).max())
sv_cat = np.linalg.svd(W_cat, compute_uv=False)
cond = float(sv_cat[0] / sv_cat[-1])

assert orth < 1e-8, f"row spaces not orthogonal: max |inner| = {orth:.2e}"
assert np.linalg.matrix_rank(Q_both) == H, "the two probes do not span the hidden state"

# do the two probes AGREE on real states?  the literal edit demands they do, exactly.
read1 = X @ Ws[0].T + bs[0]
read2 = X @ Ws[1].T + bs[1]
disagree = float(np.linalg.norm(read1 - read2, axis=1).mean())
err1 = float(np.linalg.norm(read1 - Y, axis=1).mean())
err2 = float(np.linalg.norm(read2 - Y, axis=1).mean())

display(Markdown(f"""
**Table 1 — the construction.** Held-out R² is scored against the train mean on sequences never fitted.

| quantity | value |
|---|---|
| probe 1: rank / held-out position R² | **4** / **{r2s[0]:.4f}** |
| probe 2 (on deflated states): rank / held-out position R² | **4** / **{r2s[1]:.4f}** |
| orthogonality of the two row spaces, max abs inner product | **{orth:.1e}** |
| combined row-space dimension | **{np.linalg.matrix_rank(Q_both)} of H = {H}** — the whole state |
| singular values of `A = [W₁; W₂]` | {np.array2string(sv_cat, precision=4, separator=', ')} |
| **condition number of `A`** | **{cond:,.0f}** |
| probe-1 vs probe-2 readout disagreement on **real** states | **{disagree:.3f}** sim units |
| each probe's own error vs true position | {err1:.3f} / {err2:.3f} sim units (mean ‖pos‖ = {np.linalg.norm(Y, axis=1).mean():.3f}) |

The row space is now **everything**, so "the edit direction lies outside the probe's row space" is no
longer available as an explanation. Two warnings the table already carries: `A`'s smallest singular value
is **{sv_cat[-1]:.1e}**, so inverting it amplifies by ~{1/sv_cat[-1]:,.0f}×; and the two probes disagree by
{disagree:.3f} sim units on real states, while the literal edit demands they agree **exactly**.
"""))

## §2 — The edit

Arms are built at their natural magnitude *and* **norm-matched to `‖Δh_true‖`**, because this thread has
repeatedly found that an ill-conditioned inverse buys Edit Index movement with sheer displacement
(`metric_corrected_edits` TEST 3). The norm-matched block is the apples-to-apples comparison: **same
displacement, different direction.**

One structural note that shapes the arms. Because INLP makes `row(W₁) ⊥ row(W₂)`, the **min-norm probe-1
injection already leaves probe 2's readout unchanged** — its `Δh` lies in `row(W₁)`, which `W₂` annihilates.
So "write probe 1 to the target and hold probe 2" is *numerically identical* to plain probe-1 injection, and
the only genuinely new thing the second probe can do is **demand a different value**. That is the literal
proposal, and it is the arm below.

In [ ]:
# [4] the edit setup, the arms
NE = N_EDIT
edit_obj = edits.edit_object[:NE].astype(int)
ix = np.arange(NE)
pos_e = edits.positions[:NE, :, :N_OBJ, :].astype(np.float32)
with h5py.File(edits.h5_path, "r") as f:
    vel_e = f["velocities"][:NE, :, :N_OBJ, :].astype(np.float32)

gt_roll = edits.clean_obs[:NE, EF:EF + K].astype(np.float32)
zones = build_edit_zones(pre_pos=pos_e[:, EF - 1], tgt_pos=pos_e[:, EF], pre_vel=vel_e[:, EF - 1],
                         edit_object=edit_obj, sim=sim, n_obj=N_OBJ,
                         traj_pos=pos_e[:, EF:EF + K], gt_edited_traj=gt_roll)
h0 = warm_up_to(edits.obs[:NE], EF).cpu().numpy().astype(np.float64)
target = pos_e[:, EF].reshape(NE, 4).astype(np.float64)

# the working editor, and with it the reference direction Δh_true
render_cfg = SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"],
                       x_far=sim["x_far"], n_objects=N_OBJ, radius=sim["radius"], n_frames=1, dt=dt,
                       obs_res=R, refl_min=sim["refl_min"], refl_max=sim["refl_max"],
                       fixed_reflectivities=True, obs_noise_std=0.0, boundary="open",
                       always_in_frustum=False)
refl = np.linspace(sim["refl_min"], sim["refl_max"], N_OBJ).astype(np.float32)
radii = np.full(N_OBJ, sim["radius"], np.float32)
would_be = pos_e[:, EF - 1] + vel_e[:, EF - 1] * dt
delta_pos = np.zeros((NE, N_OBJ, 2), np.float32)
delta_pos[ix, edit_obj] = pos_e[ix, EF, edit_obj] - would_be[ix, edit_obj]
hist = np.zeros((NE, N_HIST + 1, R), np.float32)
for j in range(N_HIST + 1):
    p = pos_e[:, EF - 1 - j] + delta_pos
    for i in range(NE):
        _, _, inten = render_frame(p[i], radii, refl, render_cfg)
        hist[i, j] = inten
h_oracle = teacher_force(warm_up_to(edits.obs[:NE], EF - 1 - N_HIST),
                         hist[:, :N_HIST + 1][:, ::-1]).cpu().numpy()
dh_true = h_oracle - h0
norm_true = np.linalg.norm(dh_true, axis=1, keepdims=True)

ARMS = {"Unsteered": h0, f"Observation history overwrite n={N_HIST} (oracle)": h_oracle}
ARMS["Injection · probe 1 only (rank 4 of 8)"] = (
    h0 + (target - (h0 @ Ws[0].T + bs[0])) @ np.linalg.pinv(Ws[0]).T)

resid_both = np.concatenate([target, target], 1) - (h0 @ W_cat.T + b_cat)
ARMS["Injection · BOTH probes → target (full rank 8 of 8)"] = h0 + resid_both @ np.linalg.pinv(W_cat).T

U_, sv_, Vt_ = np.linalg.svd(W_cat)
for k in (4, 5, 6, 7):
    pinv_k = Vt_[:k].T @ np.diag(1 / sv_[:k]) @ U_[:, :k].T
    ARMS[f"Injection · BOTH probes, top-{k} singular directions"] = h0 + resid_both @ pinv_k.T

# --- norm-matched block: identical displacement, different direction
NORM_MATCHED = ["Injection · probe 1 only (rank 4 of 8)",
                "Injection · BOTH probes → target (full rank 8 of 8)"]
for name in NORM_MATCHED:
    d = ARMS[name] - h0
    ARMS[f"{name} · norm-matched"] = h0 + d * norm_true / np.linalg.norm(d, axis=1, keepdims=True)
rng = np.random.default_rng(0)
rand = rng.standard_normal(h0.shape)
ARMS["Random direction · norm-matched"] = h0 + rand * norm_true / np.linalg.norm(rand, axis=1, keepdims=True)

ROLL = {k: rollout(v) for k, v in ARMS.items()}
cards = {k: edit_scorecard(v, zones, gt_roll) for k, v in ROLL.items()}
for k, c in cards.items():
    c["fidelity_ratio"] = fidelity_ratio(c, cards["Unsteered"])
    d = ARMS[k] - h0
    c["dh"] = float(np.linalg.norm(d, axis=1).mean() / np.linalg.norm(h0, axis=1).mean())
    nz = np.linalg.norm(d, axis=1) > 1e-12
    c["cos"] = float(((d[nz] * dh_true[nz]).sum(1) /
                      (np.linalg.norm(d[nz], axis=1) * np.linalg.norm(dh_true[nz], axis=1))).mean()
                     ) if nz.any() else np.nan
print(f"{len(ARMS)} arms rolled out | mean ‖h₀‖ = {np.linalg.norm(h0, axis=1).mean():.3f}, "
      f"mean ‖Δh_true‖ = {norm_true.mean():.3f}")

In [ ]:
# [5] reachability, and the scorecard
reach = {}
for name, Q in [("probe 1 only (rank 4 of 8)", Qs[0]), ("BOTH probes (rank 8 of 8)", Q_both)]:
    proj = dh_true @ Q.T @ Q
    frac = float((np.linalg.norm(proj, axis=1) / np.linalg.norm(dh_true, axis=1)).mean())
    chance = float(np.sqrt(Q.shape[0] / H))
    reach[name] = (frac, chance, frac / chance)

rows = ["| row space | reachable fraction of Δh_true | chance √(rank/H) | enrichment |", "|---|---|---|---|"]
for name, (f_, c_, e_) in reach.items():
    rows.append(f"| {name} | **{f_:.4f}** | {c_:.4f} | {e_:.2f}× |")
display(Markdown("**Table 2 — the reachability ceiling.** With both probes the ceiling is **1.0000, "
                 "exactly**: every direction, including the true edit, is available to the editor.\n\n"
                 + "\n".join(rows)))

srows = ["| arm | **Edit Index** ↑ | Target RMSE ↓ | Ghost RMSE ↓ | GT-traj RMSE ↓ | **fidelity** ↓ | "
         "‖Δh‖/‖h‖ | **cos(write, Δh_true)** ↑ |", "|---|---|---|---|---|---|---|---|"]
for k, c in cards.items():
    flag = " ⚠" if c["fidelity_ratio"] > 1.05 else ""
    ang = "" if np.isnan(c["cos"]) else f" ({np.degrees(np.arccos(np.clip(c['cos'], -1, 1))):.0f}°)"
    srows.append(f"| {k} | **{c['edit_index']:+.3f}** | {c['target_rmse']:.3f} | {c['ghost_rmse']:.3f} | "
                 f"{c['gt_traj_rmse']:.3f} | {c['fidelity_ratio']:.2f}{flag} | {c['dh']:.3f} | "
                 f"{c['cos']:+.3f}{ang} |")
display(Markdown("**Table 3 — the scorecard.** ⚠ = fidelity ratio > 1.05, i.e. the edited rollout ended "
                 "*further* from the true post-edit world than doing nothing. At H=8 an Edit Index near 0 "
                 "is usually the *garbage* reading, not the *ambiguous* one — check the RMSE columns.\n\n"
                 + "\n".join(srows)))

In [ ]:
# [6] Fig 1 — the construction and what it buys
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.3))
for ax in axes:
    style_ax(ax)

ax = axes[0]
ax.semilogy(range(1, 5), np.linalg.svd(Ws[0], compute_uv=False), "o-", color=OI["blue"], lw=2, ms=7,
            label="probe 1 (rank 4)")
ax.semilogy(range(1, 5), np.linalg.svd(Ws[1], compute_uv=False), "s-", color=OI["orange"], lw=2, ms=7,
            label="probe 2 (rank 4, on deflated states)")
ax.semilogy(range(1, H + 1), sv_cat, "D--", color=OI["grey"], lw=1.6, ms=5,
            label="stacked map A = [W₁; W₂]")
ax.set_xlabel("singular value index")
ax.set_ylabel("singular value")
ax.set_title(f"(a) spectrum — A is square but ill-conditioned (cond {cond:,.0f})", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
names = list(reach)
ax.bar(range(2), [reach[n][0] for n in names], color=[OI["sky"], OI["green"]], width=0.6,
       label="reachable fraction of Δh_true")
ax.plot(range(2), [reach[n][1] for n in names], "k_", ms=40, mew=2.5, label="chance √(rank/H)")
ax.set_xticks(range(2))
ax.set_xticklabels(["probe 1 only\n(rank 4 of 8)", "BOTH probes\n(rank 8 of 8)"], fontsize=8.5)
ax.set_ylabel("fraction of ‖Δh_true‖")
ax.set_ylim(0, 1.15)
ax.set_title("(b) the excuse is removed — the ceiling is exactly 1.0", fontsize=10)
ax.legend(fontsize=8, loc="lower right")
for i, n in enumerate(names):
    ax.text(i, reach[n][0] + 0.03, f"{reach[n][0]:.4f}", ha="center", fontsize=8.5)

ax = axes[2]
cos_arms = ["Injection · probe 1 only (rank 4 of 8)",
            "Injection · BOTH probes → target (full rank 8 of 8)",
            "Random direction · norm-matched",
            f"Observation history overwrite n={N_HIST} (oracle)"]
cos_lbl = ["Injection\nprobe 1 only", "Injection\nBOTH probes", "Random\ndirection", "Oracle\n(by construction)"]
vals = [cards[a]["cos"] for a in cos_arms]
ax.bar(range(4), vals, color=[OI["sky"], OI["green"], OI["red"], OI["grey"]], width=0.62)
for i, v in enumerate(vals):
    ax.text(i, v + 0.04, f"{v:+.3f}\n({np.degrees(np.arccos(np.clip(v, -1, 1))):.0f}°)", ha="center",
            fontsize=8)
ax.axhline(0, color="k", lw=0.9)
ax.set_xticks(range(4))
ax.set_xticklabels(cos_lbl, fontsize=8.5)
ax.set_ylabel("cos(write direction, Δh_true)")
ax.set_ylim(-0.2, 1.25)
ax.set_title("(c) …and the probe still does not point at the answer", fontsize=10)

fig.suptitle("Fig 1 — two orthogonal INLP probes spanning the whole 8-dim state: reachability vs direction "
             "(GRU · 8 hidden · dataset 4, N=256 held-out edits)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

In [ ]:
# [7] Fig 2 — the edit, at natural magnitude and at matched magnitude
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
for ax in axes:
    style_ax(ax)

ax = axes[0]
scatter_arms = [a for a in ARMS if a != "Unsteered"]
for a in scatter_arms:
    c = cards[a]
    is_or = "oracle" in a
    col = OI["green"] if is_or else (OI["red"] if c["fidelity_ratio"] > 1.05 else OI["blue"])
    ax.scatter(c["dh"], c["edit_index"], s=90 if is_or else 55, color=col, zorder=3,
               marker="*" if is_or else "o")
ax.axhline(cards["Unsteered"]["edit_index"], color=OI["grey"], lw=1.6, ls="--", label="unsteered")
ax.axvline(norm_true.mean() / np.linalg.norm(h0, axis=1).mean(), color=OI["purple"], lw=1.6, ls=":",
           label="‖Δh_true‖ — the size of a real edit")
ax.set_xscale("log")
ax.set_xlabel("‖Δh‖ / ‖h‖   (log scale)")
ax.set_ylabel("Edit Index")
ax.set_title("(a) every full-rank arm sits far right — index bought with magnitude\n"
             "(red = fidelity > 1.05, i.e. degraded; star = oracle)", fontsize=9.5)
ax.legend(fontsize=8, loc="lower left")

ax = axes[1]
bars = ["Unsteered", "Injection · probe 1 only (rank 4 of 8) · norm-matched",
        "Injection · BOTH probes → target (full rank 8 of 8) · norm-matched",
        "Random direction · norm-matched", f"Observation history overwrite n={N_HIST} (oracle)"]
blbl = ["Unsteered", "Injection · probe 1 only\n(rank 4), norm-matched",
        "Injection · BOTH probes\n(rank 8), norm-matched", "Random direction,\nnorm-matched",
        "Observation history\noverwrite (oracle)"]
vals = [cards[a]["edit_index"] for a in bars]
fid = [cards[a]["fidelity_ratio"] for a in bars]
cols = [OI["grey"], OI["sky"], OI["blue"], OI["red"], OI["green"]]
yy = np.arange(len(bars))
ax.barh(yy, vals, color=cols, height=0.66)
for i, (v, f_) in enumerate(zip(vals, fid)):
    ax.text(v + 0.02 if v >= 0 else 0.02, i, f"fid {f_:.2f}", va="center", ha="left", fontsize=8,
            color=OI["red"] if f_ > 1.05 else OI["grey"])
ax.set_yticks(yy)
ax.set_yticklabels(blbl, fontsize=8)
ax.invert_yaxis()
ax.axvline(0, color="k", lw=0.9)
ax.set_xlabel("Edit Index")
ax.set_xlim(-0.75, 0.85)
ax.set_title("(b) at IDENTICAL displacement ‖Δh_true‖ — direction is all that differs", fontsize=10)

fig.suptitle("Fig 2 — does spanning the whole hidden state make the edit land? "
             "(GRU · 8 hidden · dataset 4, N=256 held-out edits)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

## §3 — Observation space

Required by `CLAUDE.md` for any claim about an effect on the generations, through the same
`waterfall_grid(...)` helper the `history_editing/` notebooks use. The degenerate arms are shown as their
own columns — with a condition number of ~14,000 the failure mode here is collapse, and the scorecard's
near-zero Edit Index cannot distinguish that from "ambiguous" on its own.

In [ ]:
# [8] Fig 3 — the waterfall
wf = ["Unsteered", "Injection · probe 1 only (rank 4 of 8)",
      "Injection · BOTH probes → target (full rank 8 of 8)",
      "Injection · BOTH probes, top-4 singular directions",
      "Injection · BOTH probes → target (full rank 8 of 8) · norm-matched",
      "Random direction · norm-matched", f"Observation history overwrite n={N_HIST} (oracle)"]
wf_lbl = ["Unsteered", "Injection · probe 1 only\n(rank 4 of 8)", "Injection · BOTH probes\n(rank 8 of 8)",
          "Injection · BOTH probes\ntop-4 singular dirs", "Injection · BOTH probes\n(rank 8), norm-matched",
          "Random direction,\nnorm-matched", "Observation history\noverwrite (oracle)"]
labels = {a: f"{s}\nEdit Index {cards[a]['edit_index']:+.2f} · fid {cards[a]['fidelity_ratio']:.2f}"
          for a, s in zip(wf, wf_lbl)}

samples = list(np.argsort(zones.teleport)[::-1][:3])
fig = waterfall_grid(
    rolls={a: ROLL[a] for a in wf},
    ctx=edits.obs[:NE, EF - 6:EF].astype(np.float32),
    gt_roll=gt_roll,
    tgt_cx=ray_centroid(zones.target),
    ghost_cx=ray_centroid(zones.ghost),
    samples=samples,
    edit_frame=EF,
    leads_by_one=(),
    title=("Fig 3 — observation-space rollouts when the probe row space is the WHOLE hidden state\n"
           "(GRU · 8 hidden · dataset 4; the three largest teleports; each column is that arm's own "
           "free-run)"),
    labels=labels,
)
out_dir = Path("/tmp/full_rowspace_edit")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "fig3_h8_waterfall.png", dpi=125, bbox_inches="tight", facecolor="#0a0a14")
plt.show()
print("saved", out_dir / "fig3_h8_waterfall.png")

## Summary

**What this notebook measures** (invariant): whether "the edit direction lies outside the probe's row
space" is the binding constraint on readout injection, by removing that constraint exactly — two
orthogonal INLP probes whose row spaces together span the entire 8-dimensional hidden state.

### Current results (updated 2026-08-13) — GRU · 8 hidden · dataset 4, N=256 held-out edits

**1. The construction does exactly what it should.** Probe 1 and probe 2 are each **rank 4**, their row
spaces are orthogonal to **7.9e-13**, and together they span **8 of 8** dimensions. The reachable fraction
of the true edit direction rises from **0.5897** (probe 1 alone — *below* its 0.7071 chance level, the
familiar result) to **1.0000, exactly**. The excuse is gone: every direction, including the one that works,
is available to the editor.

**2. And the edit still does not land.** `cos(write, Δh_true)` = **+0.040 — an angle of 88°**, essentially
orthogonal, exactly where every other probe-derived direction in this thread sits. Probe 1 alone: −0.011
(91°). Spanning the whole state changed *what is reachable* and did not change *where the probe points*.

**3. The literal proposal is also numerically degenerate.** The stacked map `A = [W₁; W₂]` is square but
has condition number **14,373** (smallest singular value 8.8e-04). Demanding both readouts equal the target
therefore requires a write of **‖Δh‖/‖h‖ = 2137**, producing Target RMSE 660 and fidelity **261** — total
collapse, which the Edit Index reports as **+0.001**, its "equidistant *or* garbage" reading. Truncating to
the top-k singular directions walks this back (k=4: fidelity 1.58) without ever helping.
*The mechanism is stateable:* the two probes disagree by **0.923 sim units** on real states (their own
errors vs true position are 3.02 and 3.20), so demanding they agree **exactly** asks for a state more
self-consistent than any state the model actually visits.

**4. The apples-to-apples comparison — same displacement, different direction.** Scaling every write to
`‖Δh_true‖` (‖Δh‖/‖h‖ = 0.791):

| arm at identical displacement | Edit Index | fidelity |
|---|---|---|
| Unsteered | −0.489 | 1.00 |
| Injection · probe 1 only (rank 4) | −0.210 | 1.16 ⚠ |
| **Injection · BOTH probes (rank 8 — the whole state)** | **−0.277** | 1.12 ⚠ |
| Random direction | −0.172 | 1.25 ⚠ |
| **Observation history overwrite (oracle)** | **+0.529** | **0.81** |

The full-row-space write is **not better than a random direction of the same size** — it is slightly worse —
and all three probe/random arms *degrade* the rollout, while the oracle at **exactly the same displacement**
lands the edit and *improves* fidelity to 0.81. Fig 3 shows it: every injection column collapses into
saturated bands or smears, and only the oracle puts the object on the green target locator.

### Interpretation (mine, not established)

**Reachability was never the binding constraint.** The thread has repeatedly reported the row-space fraction
as the "hard ceiling" on injection-style editors, and it *is* a valid ceiling — but this notebook shows it
was never the active one. Raise it from 0.59 to 1.00 and nothing improves; the write is still 88° from the
answer and still no better than noise at matched magnitude.

What a probe gives you is a **direction that correlates with position across the data distribution**, and
inverting it answers "which state would *read* as the target", not "which state *is* the target". Those two
coincide only if the probe is an accurate model of the state→position map; at H=8 it explains R² 0.17, and
even the second, orthogonal probe only reaches 0.09. So the inverse is dominated by the 83% the probe does
not explain, and it points essentially anywhere.

Read with the companion `../history_editing/` result — the un-edited complement is **observation-shaped**
history, and writing the position history into the state does nothing a matched-norm random write does not —
the two notebooks close the same door from opposite sides. It is not that the editor cannot reach the right
state; it is that a position readout does not know where the right state is.

### Owed / scope limits

* **H=8 is a weak model** — unsteered Edit Index −0.489 (vs −0.670 at H=256), and its rollouts are visibly
  blurry in Fig 3. Its oracle reaches only +0.529 (vs +0.68). Every comparison here is internal, but the
  *degree* of degradation under editing is worse at H=8 than the thread's baseline, as `CONTROL_RUNS.md`
  already records for structural editors at H=8/H=32.
* One model, one seed, position only, linear probes only.
* **The H=256 analogue is not run.** There, spanning the full state needs 64 rank-4 probes (`iterative_probing`
  found 29 before position R² hit chance, i.e. 112 dims of linear position code). Whether a 64-probe stack
  behaves like this one is the direct follow-on, and is the version that would generalise the claim beyond
  a model small enough for two probes to exhaust.
* The near-null singular direction of `A` is what makes the literal write explode; a ridge-regularised
  formulation is a cleaner editor than truncation, and is untested.